In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.7 MB/s  0:00:00 eta 0:00:01


In [2]:
# ============================================================
# TCGA-UCEC: Download 10 Pathology Reports + Extract Text
# ============================================================

import requests
import json
import os
import fitz  

OUTPUT_DIR = "tcga_ucec_reports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Query GDC API for UCEC pathology report file IDs ──────
print("Querying GDC API...")

filters = {
    "op": "and",
    "content": [
        {"op": "=", "content": {"field": "cases.project.project_id", "value": "TCGA-UCEC"}},
        {"op": "=", "content": {"field": "data_type",               "value": "Pathology Report"}}
    ]
}

params = {
    "filters": json.dumps(filters),
    "fields":  "file_id,file_name,cases.submitter_id,cases.case_id",
    "format":  "JSON",
    "size":    10       # ← change to 500+ later for the full dataset
}

resp = requests.get("https://api.gdc.cancer.gov/files", params=params)
resp.raise_for_status()
hits = resp.json()["data"]["hits"]
print(f"Found {len(hits)} files\n")

# ── 2. Download each PDF ──────────────────────────────────────
def download_pdf(file_id, save_path):
    url = f"https://api.gdc.cancer.gov/data/{file_id}"
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# ── 3. Extract text (handles both digital PDFs and scanned ones) ──
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages_text = []
    for page in doc:
        text = page.get_text().strip()
        pages_text.append(text)
    doc.close()
    full_text = "\n".join(pages_text).strip()
    return full_text

# ── 4. Main loop ──────────────────────────────────────────────
records = []

for i, hit in enumerate(hits):
    file_id   = hit["file_id"]
    file_name = hit.get("file_name", f"{file_id}.pdf")
    case_id   = hit.get("cases", [{}])[0].get("submitter_id", "UNKNOWN")

    save_path = os.path.join(OUTPUT_DIR, f"{case_id}_{file_id}.pdf")

    print(f"[{i+1}/10] Downloading: {case_id} ... ", end="")
    try:
        download_pdf(file_id, save_path)
        text = extract_text(save_path)
        char_count = len(text)
        status = "✓ text extracted" if char_count > 100 else "⚠ scanned/empty PDF"
        print(f"{status} ({char_count} chars)")

        records.append({
            "case_id":   case_id,
            "file_id":   file_id,
            "pdf_path":  save_path,
            "text":      text,
            "char_count": char_count
        })

    except Exception as e:
        print(f"✗ failed: {e}")

print(f"\nDone. {len(records)} reports processed.")

# ── 5. Quick inspection ───────────────────────────────────────
for rec in records:
    print(f"\n{'='*60}")
    print(f"Case:  {rec['case_id']}")
    print(f"Chars: {rec['char_count']}")
    print(f"--- Preview (first 500 chars) ---")
    print(rec['text'][:500] if rec['text'] else "[NO TEXT - likely scanned PDF]")

# ── 6. Save to JSON for reuse ─────────────────────────────────
output_json = os.path.join(OUTPUT_DIR, "reports.json")
with open(output_json, "w") as f:
    json.dump(records, f, indent=2)

print(f"\nSaved all records to: {output_json}")

Querying GDC API...
Found 10 files

[1/10] Downloading: TCGA-FI-A2CX ... ✓ text extracted (8144 chars)
[2/10] Downloading: TCGA-AX-A3G3 ... ✓ text extracted (7823 chars)
[3/10] Downloading: TCGA-AX-A1CR ... ✗ failed: HTTPSConnectionPool(host='api.gdc.cancer.gov', port=443): Max retries exceeded with url: /data/3acf6cbd-a104-4bce-8ee8-cccc5a69e516 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1006)')))
[4/10] Downloading: TCGA-EY-A1G8 ... ✗ failed: HTTPSConnectionPool(host='api.gdc.cancer.gov', port=443): Max retries exceeded with url: /data/37bceb33-4bd3-4148-a44a-7ae4d141d308 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1006)')))
[5/10] Downloading: TCGA-KJ-A3U4 ... ✓ text extracted (12609 chars)
[6/10] Downloading: TCGA-BG-A0M8 ... ✓ text extracted (3749 chars)
[7/10] Downloading: TCGA-AP-A05D ... ✓ text extracted (2300 chars)
[8/10] Downloa